# Import dependencies

In [1]:
import polars_bio as pb
import time
import subprocess

INFO:polars_bio:Creating BioSessionContext


# Import data

In [2]:
data_path = "../../tests/resources/example.fastq"

In [3]:
pb.read_fastq(data_path)

INFO:polars_bio:Table: example registered for path: ../../tests/resources/example.fastq


In [4]:
start = time.time_ns()
polars_df = pb.base_sequence_quality("example")
polars_time = (time.time_ns() - start)/10**9
print(f"Polars Bio took {polars_time:.2f} seconds")
polars_df

0rows [00:00, ?rows/s]

Polars Bio took 0.07 seconds


,average,lower,median,q1,q3,upper
position,,,,,,
0,30.135,26.500000,33.083333,31.000000,34.0,38.500000
1,31.210,26.500000,34.000000,31.000000,34.0,38.500000
2,32.015,26.500000,34.000000,31.000000,34.0,38.500000
3,35.690,32.000000,37.000000,35.000000,37.0,40.000000
4,35.680,32.000000,37.000000,35.000000,37.0,40.000000
...,...,...,...,...,...,...
96,31.315,27.638889,34.000000,32.055556,35.0,39.416667
97,30.670,25.000000,34.000000,31.000000,35.0,41.000000
98,31.550,26.875000,34.000000,31.750000,35.0,39.875000


# Fastqc-rs

In [5]:
with open("report.html", "w") as f:
    start = time.time_ns()
    subprocess.run(["fqc", "-q", data_path], stdout=f)
    fqc_time = (time.time_ns() - start)/10**9
print(f"fqc took {fqc_time:.2f} seconds")

fqc took 2.03 seconds


To compare values in polars-bio with fastqc-rs you need to:
1. open report.html
2. go into base sequence quality section
3. view plots source
4. copy source into result.json

In [6]:
import json
import pandas as pd

with open("result.json", "r") as f:
    data = json.load(f)

fqc_df = pd.DataFrame(data["data"]["values"])

fqc_df = fqc_df.sort_values(by="pos").reset_index(drop=True).drop(columns=["pos"])

fqc_df.head()

,average,lower,median,q1,q3,upper
0,30.135,26.5,33.0,31.0,34.0,38.5
1,31.210,26.5,34.0,31.0,34.0,38.5
2,32.015,26.5,34.0,31.0,34.0,38.5
3,35.690,32.0,37.0,35.0,37.0,40.0
4,35.680,32.0,37.0,35.0,37.0,40.0


# Comparison

In [7]:
print(f"Polars Bio took {polars_time:.2f} seconds")
print(f"fqc took {fqc_time:.2f} seconds")
print(f"Polars Bio is {fqc_time/polars_time:.2f}x faster than fqc")

Polars Bio took 0.07 seconds
fqc took 2.03 seconds
Polars Bio is 27.17x faster than fqc


In [10]:
fqc_desc = fqc_df.describe()
fqc_desc

,average,lower,median,q1,q3,upper
count,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000
mean,35.540842,30.084158,37.524752,35.163366,38.549505,43.628713
std,2.886305,3.456114,2.375264,2.510835,2.421007,3.189458
min,30.135000,21.000000,33.000000,30.000000,34.000000,38.000000
25%,32.440000,27.500000,35.000000,33.000000,36.000000,40.500000
50%,36.770000,30.000000,38.000000,35.000000,40.000000,45.000000
75%,37.895000,33.500000,40.000000,38.000000,41.000000,45.875000
max,38.965000,35.000000,40.000000,38.000000,41.000000,48.875000


In [11]:
polars_desc = polars_df.describe()
polars_desc

,average,lower,median,q1,q3,upper
count,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000
mean,35.540842,29.970916,37.541873,35.118812,38.550743,43.698639
std,2.886305,3.486036,2.372672,2.518057,2.412681,3.173974
min,30.135000,20.611111,33.083333,30.097222,34.000000,38.000000
25%,32.440000,27.500000,35.000000,33.000000,36.000000,40.500000
50%,36.770000,30.000000,38.416667,35.000000,40.000000,44.875000
75%,37.895000,33.500000,40.000000,37.750000,41.000000,45.916667
max,38.965000,35.000000,40.000000,38.000000,41.000000,49.375000


In [12]:
fqc_desc - polars_desc

,average,lower,median,q1,q3,upper
count,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
mean,0.0,0.113243,-0.017120,0.044554,-0.001238,-0.069926
std,0.0,-0.029922,0.002593,-0.007222,0.008326,0.015484
min,0.0,0.388889,-0.083333,-0.097222,0.000000,0.000000
25%,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.0,0.000000,-0.416667,0.000000,0.000000,0.125000
75%,0.0,0.000000,0.000000,0.250000,0.000000,-0.041667
max,0.0,0.000000,0.000000,0.000000,0.000000,-0.500000
